## Langchain from scratch

### 1. Early langchain

lets create a dummy llm ( to emulate an langchain LLM component )

In [2]:
import random

class DummyLLM:

  def __init__(self):
    print('LLM created')

  def predict(self, prompt):

    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}

Also, lets create a dummy prompt template class

In [1]:
class DummyPromptTemplate:

  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  def format(self, input_dict):
    return self.template.format(**input_dict)

now, lets create a prompt with the above defined class

In [3]:
template = DummyPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

prompt = template.format({'length':'short','topic':'india'})

print(prompt)


Write a short poem about india


Lets create an llm object with the above defined class and pass this prompt object to this one.

In [4]:
llm = DummyLLM()

llm.predict(prompt)

LLM created


{'response': 'Delhi is the capital of India'}

### 2. Langchain when they realized they need to create some chains etc

To combine the above LLM and prompt in to one thing , they created LLM-chain component right, lets create a dummy one to emulate the same here.

In [5]:
class DummyLLMChain:

    #for this the inputs will be obviously llm and prompt
  def __init__(self, llm, prompt):
    self.llm = llm
    self.prompt = prompt

  def run(self, input_dict):

    #the manual thing which we are doing above , we are simply doing here
    final_prompt = self.prompt.format(input_dict)
    result = self.llm.predict(final_prompt)

    #We are finally returning the response
    return result['response']


Lets, emulate the workflow with this new LLM-chain that we created

In [6]:
#create a prompt object ( no need to format )
template = DummyPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

#create a LLm object, no need to predict
llm = DummyLLM()

#create a chain, which combines above 2 objects
chain = DummyLLMChain(prompt=template, llm=llm)

#now run the chain
chain.run({'length':'short', 'topic': 'india'})

LLM created


'Delhi is the capital of India'

### 3. Langchain standardized with runnables

1. why do we need this? 

    Here, the basic need is to standardize all the individual components right, for that we will create an abstract class and then make all of the components inherit this class , then all will be forced to follow the same standard as the parent abstract class. So, this is the way to make sure everyone who designs new langchain component will make it standard.

2. How does this work?

    Here, the main job of abstract class is that, whatever functions present in this class, need to defined in the child class which inherits it, and they need to complete this incomplete functions inside them mandatory.

    Here, we will keep invoke() funcition as abstract method, in the parent class, then if the child which inherits this parent, wont be able to execute itself, unless and until it defined this abstract method within itself.

In [7]:
from abc import abstractmethod, ABC

class Runnable(ABC):

  @abstractmethod
  def invoke(input_data):
    pass

here, we will inherit the abstract class, but if we dont define abstract method present inside it, we will get an error as shown below.

In [9]:
import random

class DummyLLM(Runnable):

  def __init__(self):
    print('LLM created')

  def predict(self, prompt):

    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}
  
#Lets try to create an object\
llm = DummyLLM()

TypeError: Can't instantiate abstract class DummyLLM without an implementation for abstract method 'invoke'

See, the above it says, define invoke() method.

So, lets define it and put everything which is inside the predict method in it.

In [11]:
import random

class DummyLLM(Runnable):

    def __init__(self):
        print('LLM created')

    def invoke(self, prompt):

        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for Artificial Intelligence'
        ]

        return {'response': random.choice(response_list)} 

# We can remove this method, but if many people who are using in their applications will break their code
# instead we will put a disclaimer that it will be removed in the future etc
    def predict(self, prompt):

        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for Artificial Intelligence'
        ]

        print("This method will be deprecated in the future, please call invoke")
        return {'response': random.choice(response_list)}   
  
#Lets try to create an object\
llm = DummyLLM()

LLM created


See, now it works.

Lets do the same for all the other dummmy components also which we have created before.

In [14]:
#create a standard prompt template
class DummyPromptTemplate(Runnable):

  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  # This is new standard method
  def invoke(self, input_dict):
    return self.template.format(**input_dict)

  #This is the old method with the disclaimer message
  def format(self, input_dict):
    print("This method will be deprecated please use invoke() in the future")
    return self.template.format(**input_dict)


# create a dummy standard string output parser
class DummyStrOutputParser(Runnable):

  def __init__(self):
    pass

  def invoke(self, input_data):
    return input_data['response']

Now, lets create a standard runnable connector which is also a runnable, but it will take the output of 1 runnable and give it as input to another runnable and give us the output.

In [13]:
class DummyRunnableConnector(Runnable):

  def __init__(self, runnable_list):
    self.runnable_list = runnable_list

  def invoke(self, input_data):

    #Here as all the runnables are passed a list, which means they need to executed in order
    # the output of the first element will be input of the second and so on till the end of the list
    for runnable in self.runnable_list:
      input_data = runnable.invoke(input_data)

    #finally return the output data
    return input_data

Now, lets connect everything together and make chains with the new standard runnable connector

In [15]:
#create prompt
template = DummyPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

#Create LLM
llm = DummyLLM()

#Create output parser
parser = DummyStrOutputParser()

#Lets create a chain with all these now
chain = DummyRunnableConnector([template,llm,parser])

#Lets execute the chain, by giving the input of the 1st runnable in the list which is prompt
#This should give the output of the last runnable in the list which is stroutputparser
chain.invoke({'length':'long', 'topic': 'India'})

LLM created


'Delhi is the capital of India'

yes, we got it and it is sucessful

#### 3.1 Lets create a slightly more complex example, where we create 2 chains, combine those 2 chains to create a mega chain which is also a runnable

Lets define all the components (lego blocks) first

In [16]:
template1= DummyPromptTemplate(
    template="Write a joke about {topic}",
    input_variables=['topic']
)

template2 = DummyPromptTemplate(
    template="Explain the following joke {response}",
    input_variables=['response']
)

llm = DummyLLM()

parser = DummyStrOutputParser()

LLM created


lets create chains ( connect the blocks together )

In [17]:
chain1= DummyRunnableConnector([template1, llm])

#Here as we are not using output parser, we will get dictionary with key "response"
chain1.invoke({'topic': 'AI'})


{'response': 'AI stands for Artificial Intelligence'}

Here, as we are planning to connect chain2 after chain1, we need to make sure output of the chain 1 is compatible with input of chain2, that's why we defined, the template2 to expect a dictionary with 'response' key, also that's why we are not using parser in chain1

In [19]:

chain2= DummyRunnableConnector([template2, llm , parser])

#Lets test it by giving input in the format of template2 runnabnle since that is 1st in the list
chain2.invoke({'response': "This is a joke"})

'AI stands for Artificial Intelligence'

These are working individually, but the goal is to connect them and then invoke them with the mega chain right, lets do that.

In [ ]:
final_chain = DummyRunnableConnector([chain1,chain2])

#lets invoke this by passing the input format of chain1, we will get the output in the format of chain2
final_chain.invoke({'topic': "AI"})

'AI stands for Artificial Intelligence'

Hence, we sucessfully designed langchain from scratch